# Nghiên cứu và EDA cho quy luật sinh dữ liệu mua hàng

**Mục tiêu:** xác định cách mô phỏng khách hàng, giỏ hàng, nhu cầu sản phẩm, thời điểm mua và hậu mãi cho nghiệp vụ bán lẻ của dự án.

Mỗi nhóm đi từ **câu hỏi → căn cứ nghiên cứu → EDA → đánh giá → quy luật → ứng dụng**. Kết quả định lượng mô tả đúng nguồn/cohort; bằng chứng Việt Nam bổ sung bối cảnh, không tự biến thành tỷ lệ theo đơn tại shop.

| Dữ liệu | Đơn vị sử dụng | Phạm vi phân tích |
|---|---|---|
| UCI UK | Invoice, mã hàng, khách có CustomerID | `[2010-12-01, 2011-12-01)`; có khách sỉ |
| H&M | Dòng mua sản phẩm, ngày mua/khách | Năm 2019; ngày mua không phải số đơn |
| Olist Brazil | Đơn, sản phẩm trong đơn, phiếu đánh giá | Năm 2017; product không phải variant |
| eCommerce Behavior | Sự kiện và phiên | Tháng 10/2019, có biên 1/11; phụ lục tham khảo |

Bảng cuối nối kết quả với quy tắc lấy mẫu và bảng nghiệp vụ. R03 đã được áp dụng qua trọng số nhóm thứ hạng H&M; các tham số chưa đủ căn cứ giữ từ kịch bản cũ và được ghi là giả định trong [danh mục chỉ số](../../docs/project/GENERATOR_METRICS.md). Chi tiết nguồn Việt Nam ở [báo cáo nghiên cứu](../../docs/project/GENERATOR_VIETNAM_RESEARCH.md).

In [ ]:
%matplotlib inline
from math import ceil
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent.parent

data_dir = repo_root / "data" / "generator" / "eda" / "raw"

## 0. Dữ liệu và điều kiện đo

Kiểm tra phạm vi, giá trị thiếu và cohort trước khi so sánh. Mỗi nguồn được phân tích độc lập; không nối định danh giữa các dataset.

### 0.1. Olist — phạm vi và dữ liệu thiếu

In [ ]:
orders = pl.scan_csv(
    data_dir / "olist" / "olist_orders_dataset.csv", infer_schema=False,
).select(
    "order_id", "customer_id", "order_status", "order_purchase_timestamp",
    "order_delivered_customer_date", "order_estimated_delivery_date",
).with_columns(
    pl.col("order_purchase_timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
    pl.col("order_delivered_customer_date").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
    pl.col("order_estimated_delivery_date").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
).collect(engine="streaming")

In [ ]:
orders.head()

In [ ]:
orders.select(
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date",
).describe()

**Nhận xét:** Nguồn bao phủ 2016–2018; cohort năm 2017 nằm trọn trong khoảng thời gian quan sát.

In [ ]:
orders.null_count()

**Nhận xét:** Có 2.965 đơn thiếu ngày giao thực tế; phép đo giao nhận chỉ dùng đơn đủ ngày, không điền thời gian thiếu thành 0.

In [ ]:
orders.select(pl.col("order_id").is_duplicated().sum().alias("duplicate_order_id"))

**Nhận xét:** Không trùng order_id trong bảng orders; mỗi khóa tương ứng một đơn trước khi nối bảng con.

### 0.2. H&M — phạm vi và dữ liệu thiếu

In [ ]:
hm_purchases = pl.scan_csv(
    data_dir / "hm" / "transactions_train.csv", infer_schema=False,
).select(
    "t_dat", "customer_id", "article_id", "price", "sales_channel_id",
).with_columns(
    pl.col("t_dat").str.to_date("%Y-%m-%d", strict=False),
    pl.col("price").cast(pl.Float64, strict=False),
)

In [ ]:
hm_purchases.select("price", "t_dat").collect(engine="streaming").describe()

**Nhận xét:** Khoảng dữ liệu chứa trọn năm 2019; giá được giữ theo thang đo nguồn để tránh suy ra giá VND không có căn cứ.

In [ ]:
hm_purchases.null_count().collect(engine="streaming")

**Nhận xét:** Các cột đang khảo sát không thiếu giá trị; chưa cần loại dòng vì null ở bước đọc H&M.

### 0.3. UCI — phạm vi và cohort giao dịch mua

In [ ]:
uci_transactions = pl.read_csv(
    data_dir / "uci" / "online_retail.csv", infer_schema=False,
).with_columns(
    pl.col("Quantity").cast(pl.Int64, strict=False),
    pl.col("UnitPrice").cast(pl.Float64, strict=False),
    pl.col("InvoiceDate").str.to_datetime("%Y-%m-%dT%H:%M:%S", strict=False),
)

uci_transactions.head()

In [ ]:
uci_transactions.select("Quantity", "UnitPrice", "InvoiceDate").describe()

**Nhận xét:** Nguồn có Quantity và UnitPrice âm; cần tách giao dịch mua dương trước khi rút phân phối giỏ và số lượng.

In [ ]:
uci_transactions.null_count()

**Nhận xét:** CustomerID thiếu ở 135.080 dòng; các dòng này vẫn có ích cho giỏ hàng nhưng không đủ định danh để đo mua lại.

UK, `[2010-12-01, 2011-12-01)`, giữ giờ nguồn. Invoice phải có thời điểm/quốc gia thống nhất và tối đa một CustomerID khác null. Chỉ lấy mua có Quantity/UnitPrice dương, bỏ invoice bắt đầu bằng C và mã hàng ngoài dạng 5 chữ số + tối đa 2 chữ cái.

Giữ dòng trùng và các giá trị lớn. Invoice thiếu CustomerID vẫn dùng cho giỏ/số lượng, chỉ loại khỏi phép đo mua lại.

In [ ]:
uci_invoice_metadata = uci_transactions.group_by("InvoiceNo").agg(
    invoice_at=pl.col("InvoiceDate").min(),
    invoice_times=pl.col("InvoiceDate").n_unique(),
    customer_id=pl.col("CustomerID").drop_nulls().first(),
    customer_ids=pl.col("CustomerID").drop_nulls().n_unique(),
    country=pl.col("Country").first(),
    countries=pl.col("Country").n_unique(),
)

uci_eligible_invoices = uci_invoice_metadata.filter(
    pl.col("InvoiceNo").is_not_null()
    & ~pl.col("InvoiceNo").str.to_uppercase().str.starts_with("C")
    & pl.col("invoice_at").is_between(pl.datetime(2010, 12, 1), pl.datetime(2011, 12, 1), closed="left")
    & (pl.col("invoice_times") == 1)
    & (pl.col("customer_ids") <= 1)
    & (pl.col("countries") == 1)
    & (pl.col("country") == "United Kingdom"),
)

In [ ]:
uci_sales_lines = uci_transactions.join(
    uci_eligible_invoices.select("InvoiceNo"), on="InvoiceNo", how="inner", validate="m:1",
).filter(
    (pl.col("Quantity") > 0)
    & (pl.col("UnitPrice") > 0)
    & pl.col("StockCode").str.contains(r"^[0-9]{5}[A-Za-z]{0,2}$"),
)

uci_order_items = uci_sales_lines.group_by("InvoiceNo", "StockCode").agg(
    quantity=pl.col("Quantity").sum(),
    item_value=(pl.col("Quantity") * pl.col("UnitPrice")).sum(),
)
uci_orders = uci_order_items.group_by("InvoiceNo").agg(
    sku_count=pl.len(),
    units=pl.col("quantity").sum(),
    goods_value=pl.col("item_value").sum(),
).join(
    uci_eligible_invoices.select("InvoiceNo", "invoice_at", "customer_id"),
    on="InvoiceNo", how="left", validate="1:1",
)

uci_orders.select(
    orders=pl.len(),
    known_customer_orders=pl.col("customer_id").is_not_null().sum(),
    customers=pl.col("customer_id").drop_nulls().n_unique(),
)

**Nhận xét:** Cohort có 17.118 đơn, trong đó 15.847 đơn xác định được khách; hai phép đo giỏ và mua lại có mẫu số khác nhau.

## 1. Khách hàng — tần suất và mua lại

**Câu hỏi:** khách mua bao nhiêu lần, khoảng cách mua có khác giữa nhóm mua ít và nhiều; một khách có thể mua nhiều kênh không?

**Căn cứ:** [TGM 2025, S02](https://tgmresearch.com/press-tgm-vietnam-ecommerce-insights-2025.html) công bố 50% mua online hằng tuần trên mẫu 1.120 người 18–54 tuổi; đây là tần suất nhiều nơi bán. [PwC 2024, S03](https://www.pwc.com/vn/vn/publications/vietnam-publications/voice-of-consumer-2024.html), mẫu VN 515 người, ghi nhận mua qua điện thoại và cửa hàng cùng phổ biến; các nhóm có thể trùng nhau.

**Thực nghiệm:** đối chiếu UCI ở cấp invoice và H&M ở cấp ngày mua; không dùng tỷ lệ đa kênh của người trả lời làm tỷ trọng đơn online/POS.

### 1.1. Thực nghiệm UCI — số đơn và khoảng mua

Một đơn là một InvoiceNo; chỉ khách có CustomerID. Giữ các đơn cùng ngày/cùng thời điểm. Nhóm 1, 2–4 và ≥5 đơn tóm tắt tần suất trong 12 tháng, không phải nhãn trung thành.

In [ ]:
uci_buyer_orders = uci_orders.filter(pl.col("customer_id").is_not_null())
uci_customer_frequency = uci_buyer_orders.group_by("customer_id").len(name="orders")
uci_frequency_distribution = uci_customer_frequency.group_by("orders").len(name="customers").sort(
    "orders",
).with_columns(
    probability=pl.col("customers") / pl.col("customers").sum(),
)

uci_customer_frequency.select(
    customers=pl.len(),
    repeat_customer_pct=100.0 * (pl.col("orders") >= 2).mean(),
    orders_p50=pl.col("orders").median(),
    orders_p90=pl.col("orders").quantile(0.9),
)

**Nhận xét:** 63,99% trong 3.880 khách xác định được có ít nhất hai đơn; số đơn trung vị là 2, phân vị 90% là 8.

In [ ]:
uci_customer_groups = uci_customer_frequency.with_columns(
    order_group=pl.when(pl.col("orders") == 1).then(pl.lit("1 đơn"))
    .when(pl.col("orders") <= 4).then(pl.lit("2–4 đơn"))
    .otherwise(pl.lit("≥5 đơn")),
)
uci_frequency_groups = uci_customer_groups.group_by("order_group").agg(
    customers=pl.len(),
    orders_p50=pl.col("orders").median(),
).sort("order_group").with_columns(
    share_pct=100.0 * pl.col("customers") / pl.col("customers").sum(),
)

uci_frequency_groups.with_columns(pl.col("share_pct").round(2))

**Nhận xét:** Nhóm một đơn chiếm 36,01%, 2–4 đơn chiếm 39,51%, ≥5 đơn chiếm 24,48%; đây là tần suất theo invoice trong 12 tháng.

In [ ]:
uci_purchase_gaps = uci_buyer_orders.sort("customer_id", "invoice_at", "InvoiceNo").with_columns(
    previous_at=pl.col("invoice_at").shift(1).over("customer_id"),
).filter(pl.col("previous_at").is_not_null()).with_columns(
    gap_days=(pl.col("invoice_at") - pl.col("previous_at")).dt.total_seconds() / 86400,
    same_day=pl.col("invoice_at").dt.date() == pl.col("previous_at").dt.date(),
).join(
    uci_customer_groups.select("customer_id", "order_group"),
    on="customer_id", how="left", validate="m:1",
)

uci_gap_profile = uci_purchase_gaps.group_by("order_group").agg(
    gaps=pl.len(),
    gap_p50=pl.col("gap_days").median(),
    gap_p90=pl.col("gap_days").quantile(0.9),
    same_day_pct=100.0 * pl.col("same_day").mean(),
).sort("order_group")

uci_gap_profile.with_columns(pl.col("gap_p50", "gap_p90", "same_day_pct").round(2))

**Nhận xét:** Khoảng mua trung vị là 63,00 ngày ở nhóm 2–4 đơn và 17,92 ngày ở nhóm ≥5 đơn; cả hai nhóm đều có nhiều đơn cùng ngày.

In [ ]:
uci_gap_distribution = uci_purchase_gaps.group_by("order_group", "gap_days").len(
    name="gaps",
).sort("order_group", "gap_days").with_columns(
    probability=pl.col("gaps") / pl.col("gaps").sum().over("order_group"),
).with_columns(
    cumulative_probability=pl.col("probability").cum_sum().over("order_group"),
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
for group in ("2–4 đơn", "≥5 đơn"):
    curve = uci_gap_distribution.filter(pl.col("order_group") == group)
    ax.step(curve["gap_days"], curve["cumulative_probability"], where="post", label=group)
ax.set(xlabel="Khoảng cách giữa hai đơn (ngày)", ylabel="Tỷ lệ khoảng mua tích lũy", title="Khoảng mua theo nhóm — UCI UK")
ax.legend()
fig.tight_layout()
plt.show()

**Nhận xét:** Nhóm mua nhiều đơn có đường tích lũy dịch về khoảng chờ ngắn hơn; 12,27% khoảng giữa hai đơn liên tiếp thuộc cùng ngày.

**Đánh giá:** UCI đo được số đơn trực tiếp và giữ nhiều invoice cùng ngày; phân phối này vẫn thuộc khách UK của nguồn có bán sỉ.

### 1.2. Thực nghiệm H&M — số ngày mua và khoảng mua

Khảo sát năm 2019. Một lần quan sát là một ngày khách có mua; mẫu số chỉ gồm khách đã mua trong năm. Dùng nguồn này để đối chiếu quan hệ tần suất–khoảng mua, không thay số ngày bằng số đơn.

In [ ]:
hm_2019 = hm_purchases.filter(pl.col("t_dat").dt.year() == 2019)

purchase_days = hm_2019.select("customer_id", "t_dat").unique().collect(engine="streaming")
customer_frequency = purchase_days.group_by("customer_id").len(name="purchase_days")

customer_frequency.select("purchase_days").describe()

**Nhận xét:** Khách có trung vị 3 ngày mua nhưng trung bình 4,72; một nhóm mua thường xuyên kéo trung bình lên.

In [ ]:
customer_frequency.select(
    repeat_customer_pct=((pl.col("purchase_days") > 1).mean() * 100).round(2),
)

**Nhận xét:** 66,20% khách đã mua có ít nhất hai ngày mua trong năm; tỷ lệ này không bao gồm người chưa phát sinh mua.

In [ ]:
purchase_gaps = purchase_days.sort("customer_id", "t_dat").with_columns(
    gap_days=pl.col("t_dat").diff().over("customer_id").dt.total_days(),
).filter(pl.col("gap_days").is_not_null())

purchase_gaps.select("gap_days").describe()

**Nhận xét:** Khoảng mua trung bình 35,26 ngày vượt trung vị 19 ngày; dùng một khoảng chờ cố định sẽ bỏ qua độ phân tán này.

**Phân phối tần suất:** Đếm số ngày mua/khách trong năm 2019, dùng số khách làm trọng số. Nhóm 1, 2–4 và ≥5 ngày chỉ để tóm tắt, không gán nhãn trung thành cho khách.

In [ ]:
frequency_distribution = customer_frequency.group_by("purchase_days").len(
    name="customers",
).sort("purchase_days").with_columns(
    probability=pl.col("customers") / pl.col("customers").sum(),
)

customer_groups = customer_frequency.with_columns(
    frequency_group=pl.when(pl.col("purchase_days") == 1).then(pl.lit("1 ngày"))
    .when(pl.col("purchase_days") <= 4).then(pl.lit("2–4 ngày"))
    .otherwise(pl.lit("≥5 ngày")),
)

frequency_groups = customer_groups.group_by("frequency_group").agg(
    customers=pl.len(),
    days_p50=pl.col("purchase_days").median(),
    days_p90=pl.col("purchase_days").quantile(0.9),
).sort("frequency_group").with_columns(
    share_pct=100.0 * pl.col("customers") / pl.col("customers").sum(),
)

frequency_groups.with_columns(pl.col("share_pct").round(2))

**Nhận xét:** Ba nhóm 1, 2–4 và ≥5 ngày mua chiếm lần lượt 33,80%, 32,65% và 33,55%; đây là nhóm tần suất quan sát, chưa phải nhãn trung thành.

**Khoảng mua theo nhóm tần suất:** Lấy xác suất từng khoảng ngày từ các cặp ngày mua liên tiếp trong cùng nhóm. Các phân vị là ngày đầu tiên đạt xác suất tích lũy tương ứng; nhóm một ngày mua không có khoảng mua lại.

In [ ]:
gap_distribution = purchase_gaps.join(
    customer_groups.select("customer_id", "frequency_group"),
    on="customer_id", how="inner", validate="m:1",
).group_by("frequency_group", "gap_days").len(name="gaps").sort(
    "frequency_group", "gap_days",
).with_columns(
    probability=pl.col("gaps") / pl.col("gaps").sum().over("frequency_group"),
).with_columns(
    cumulative_probability=pl.col("probability").cum_sum().over("frequency_group"),
)

gap_profile = gap_distribution.group_by("frequency_group").agg(
    gaps=pl.col("gaps").sum(),
    gap_p50=pl.col("gap_days").filter(pl.col("cumulative_probability") >= 0.5).min(),
    gap_p75=pl.col("gap_days").filter(pl.col("cumulative_probability") >= 0.75).min(),
    gap_p90=pl.col("gap_days").filter(pl.col("cumulative_probability") >= 0.9).min(),
    gap_p95=pl.col("gap_days").filter(pl.col("cumulative_probability") >= 0.95).min(),
).sort("frequency_group")

gap_profile

**Nhận xét:** Nhóm ≥5 ngày mua có khoảng mua trung vị 17 ngày, ngắn hơn mức 52 ngày ở nhóm 2–4 ngày.

**Đánh giá:** H&M cho phép mô tả khác biệt giữa các nhóm ngày mua; chưa đo số đơn trong một ngày.

**Đánh giá giữa nguồn:** cả hai cho thấy khoảng mua của nhóm tần suất cao ngắn hơn; UCI giữ được nhiều đơn cùng ngày, H&M không tách được các đơn đó. Khác tỷ lệ mua lại chưa chứng minh khác hành vi giữa hai thị trường vì cohort và đơn vị đo khác nhau.

**Quy luật R01:** giữ riêng `uci_frequency_distribution`/`uci_gap_distribution` và `frequency_distribution`/`gap_distribution`. Đề xuất khoảng mua phụ thuộc nhóm tần suất; không chọn số đơn và khoảng mua độc lập rồi khẳng định giữ nguyên phân phối.

**Ứng dụng:** lịch `orders` theo `customer_id`; chỉ UCI là tham chiếu trực tiếp số đơn. Tỷ trọng nhóm, số đơn/khách và phân bổ online/POS tại shop VN chưa chốt. Khi giới hạn lịch sử hoặc tổng đơn, phải ghi rõ việc điều chỉnh có thể thay đổi phân phối tham chiếu.

## 2. Giỏ hàng — số mặt hàng, số lượng và giá trị

**Câu hỏi:** một đơn có bao nhiêu mặt hàng, mua bao nhiêu đơn vị mỗi mặt hàng và giá trị đơn hình thành thế nào?

**Căn cứ:** [KSMS 2024, S01](https://www.nso.gov.vn/tin-tuc-thong-ke/2025/05/thong-cao-bao-chi-ket-qua-khao-sat-muc-song-dan-cu-nam-2024/) cho thấy khác biệt chi tiêu giữa các nhóm dân cư; tổng chi tiêu/tháng không phải ngân sách tại shop. [Metric 2025, S07](https://metric.vn/insights/cong-bo-bao-cao-thuong-mai-dien-tu-viet-nam-2025/) công bố dải giá theo ngành và tỷ trọng doanh số; chưa cho phân phối số lượng theo đơn.

**Thực nghiệm:** Olist đo product/đơn, UCI đo StockCode/đơn và Quantity. So sánh để chọn cách mô hình hóa, không quy đổi tiền nguồn sang VND hoặc mặc định product/StockCode là variant.

### 2.1. Thực nghiệm Olist — giỏ hàng và giá trị đơn

Năm 2017. Tiền hàng, phí vận chuyển và thanh toán được tính riêng; bảng con gộp về mỗi đơn trước khi join. Đơn thiếu dòng hàng giữ null, không coi là giỏ rỗng.

In [ ]:
orders_2017 = orders.filter(pl.col("order_purchase_timestamp").dt.year() == 2017)

items = pl.scan_csv(
    data_dir / "olist" / "olist_order_items_dataset.csv", infer_schema=False,
).select("order_id", "product_id", "price", "freight_value").with_columns(
    pl.col("price", "freight_value").cast(pl.Float64, strict=False),
).collect(engine="streaming")

payments = pl.scan_csv(
    data_dir / "olist" / "olist_order_payments_dataset.csv", infer_schema=False,
).select("order_id", "payment_type", "payment_value").with_columns(
    pl.col("payment_value").cast(pl.Float64, strict=False),
).collect(engine="streaming")

reviews = pl.scan_csv(
    data_dir / "olist" / "olist_order_reviews_dataset.csv", infer_schema=False,
).select("order_id", "review_score").with_columns(
    pl.col("review_score").cast(pl.Float64, strict=False),
).collect(engine="streaming")

In [ ]:
order_items = items.group_by("order_id").agg(
    item_lines=pl.len(),
    products=pl.col("product_id").n_unique(),
    goods_value=pl.col("price").sum(),
    freight_value=pl.col("freight_value").sum(),
)
order_payments = payments.group_by("order_id").agg(
    payment_total=pl.col("payment_value").sum(),
)
order_reviews = reviews.group_by("order_id").agg(
    review_score=pl.col("review_score").mean(),
)

order_metrics = orders_2017.join(
    order_items, on="order_id", how="left", validate="1:1",
).join(
    order_payments, on="order_id", how="left", validate="1:1",
).join(
    order_reviews, on="order_id", how="left", validate="1:1",
)

order_metrics.select(
    "item_lines", "products", "goods_value", "freight_value", "payment_total",
).describe()

**Nhận xét:** Tiền hàng trung bình 138,09 cao hơn trung vị 84,99; 522 đơn thiếu dòng hàng được giữ null thay vì biến thành giỏ rỗng.

In [ ]:
basket_distribution = order_metrics.filter(pl.col("products").is_not_null()).group_by(
    "products",
).len(name="orders").sort("products").with_columns(
    share_pct=100.0 * pl.col("orders") / pl.col("orders").sum(),
)

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(
    x=basket_distribution["products"].to_list(), y=basket_distribution["share_pct"].to_numpy(),
    color="#4C72B0", ax=ax,
)
ax.set(xlabel="Số sản phẩm khác nhau/đơn", ylabel="% đơn có dòng hàng", title="Giỏ hàng — Olist 2017")
fig.tight_layout()
plt.show()

**Nhận xét:** Giỏ một sản phẩm khác nhau áp đảo; số sản phẩm trong giỏ cần phân phối lệch về các giá trị nhỏ.

**Phân phối để sinh giỏ:** Xác suất số sản phẩm tính trên đơn có dòng hàng. Số dòng cùng `product_id` trong đơn dùng làm đại diện số lượng ở cấp sản phẩm nguồn; chưa tương đương số lượng mỗi variant.

In [ ]:
basket_distribution.select("products", "orders", pl.col("share_pct").round(2))

**Nhận xét:** Đơn có ≥5 sản phẩm rất ít; các tỷ lệ hiển thị 0,00% do làm tròn không có nghĩa số đơn bằng 0.

In [ ]:
order_product_lines = items.join(
    orders_2017.select("order_id"), on="order_id", how="inner", validate="m:1",
).group_by("order_id", "product_id").len(name="lines_per_product")

quantity_distribution = order_product_lines.group_by("lines_per_product").len(
    name="order_products",
).sort("lines_per_product").with_columns(
    probability=pl.col("order_products") / pl.col("order_products").sum(),
)

quantity_distribution

**Nhận xét:** 93,11% cặp đơn–sản phẩm có một dòng hàng; số dòng cùng sản phẩm cần được phân biệt với số sản phẩm khác nhau trong giỏ.

**Giá trị đơn theo kích thước giỏ:** Tiền hàng theo đơn vị tiền nguồn, chưa gồm phí vận chuyển; chỉ đơn có dòng hàng.

In [ ]:
basket_value_profile = order_metrics.filter(pl.col("products").is_not_null()).group_by(
    "products",
).agg(
    orders=pl.len(),
    goods_p50=pl.col("goods_value").median(),
    goods_p90=pl.col("goods_value").quantile(0.9),
    lines_p50=pl.col("item_lines").median(),
).sort("products")

basket_value_profile.with_columns(pl.col("goods_p50", "goods_p90").round(2))

**Nhận xét:** Tiền hàng trung vị của giỏ một và hai sản phẩm lần lượt là 83,99 và 129,90 theo tiền nguồn; nhóm giỏ lớn có rất ít mẫu.

**Quy luật đề xuất:** Chọn kích thước giỏ và số lượng theo hai phân phối riêng; giá trị đơn hình thành từ sản phẩm và số lượng. Phân biệt product nguồn với variant của repo.

### 2.2. Thực nghiệm UCI — số mã hàng trong một đơn

Gộp các dòng cùng `(InvoiceNo, StockCode)` trước khi đếm mã/đơn. Bảng xác suất hiển thị 10 kích thước phổ biến; đường tích lũy dùng toàn bộ đơn.

In [ ]:
uci_orders.select("sku_count", "units", "goods_value").describe()

**Nhận xét:** Số mã/đơn trung bình 26,15 cao hơn trung vị 15; các giỏ rất lớn làm phân phối lệch phải.

In [ ]:
uci_basket_distribution = uci_orders.group_by("sku_count").len(name="orders").sort(
    "sku_count",
).with_columns(
    probability=pl.col("orders") / pl.col("orders").sum(),
)

uci_basket_distribution.sort("orders", descending=True).head(10)

**Nhận xét:** Giỏ một mã là mức riêng lẻ phổ biến nhất nhưng chỉ chiếm 7,83%; không thể xem đây là kiểu giỏ của đa số đơn.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.ecdfplot(x=uci_orders["sku_count"].to_numpy(), ax=ax)
ax.set_xscale("log")
ax.set(xlabel="Số mã hàng/đơn (thang log)", ylabel="Tỷ lệ đơn tích lũy", title="Giỏ hàng — UCI UK")
fig.tight_layout()
plt.show()

**Nhận xét:** 50% đơn có tối đa 15 mã hàng, 90% có tối đa 52 mã; phần đuôi giỏ lớn vẫn hiện rõ trên thang log.

**Đánh giá:** Giỏ UCI lớn hơn Olist; khác thị trường, ngành hàng và định nghĩa mã hàng nên không gộp hai phân phối.

### 2.3. Thực nghiệm UCI — số lượng và giá trị theo giỏ

Quantity cộng theo mã hàng trong từng đơn, không đếm số dòng thay số lượng. Giá trị đơn là tổng Quantity × UnitPrice, đơn vị GBP. Nhóm giỏ chỉ để tóm tắt, không cắt dữ liệu.

In [ ]:
uci_quantity_distribution = uci_order_items.group_by("quantity").len(name="order_skus").sort(
    "quantity",
).with_columns(
    probability=pl.col("order_skus") / pl.col("order_skus").sum(),
)

uci_order_items.select("quantity", "item_value").describe()

**Nhận xét:** Quantity trung bình 9,71 cao hơn trung vị 3, với giá trị tối đa 74.215; đuôi lớn cần được giữ rõ khi diễn giải nguồn có khách sỉ.

In [ ]:
uci_basket_value = uci_orders.with_columns(
    basket_group=pl.when(pl.col("sku_count") == 1).then(pl.lit("1 mã"))
    .when(pl.col("sku_count") <= 4).then(pl.lit("2–4 mã"))
    .when(pl.col("sku_count") <= 9).then(pl.lit("5–9 mã"))
    .otherwise(pl.lit("≥10 mã")),
).group_by("basket_group").agg(
    orders=pl.len(),
    units_p50=pl.col("units").median(),
    goods_p50=pl.col("goods_value").median(),
    goods_p90=pl.col("goods_value").quantile(0.9),
).sort("basket_group")

uci_basket_value.with_columns(pl.col("goods_p50", "goods_p90").round(2))

**Nhận xét:** Giá trị đơn trung vị tăng từ 40 GBP ở giỏ một mã lên 334,54 GBP ở giỏ ≥10 mã; số mã và tổng giá trị có liên hệ trong cohort.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.ecdfplot(x=uci_order_items["quantity"].to_numpy(), ax=ax)
ax.set_xscale("log")
ax.set(xlabel="Số lượng/mã hàng trong đơn (thang log)", ylabel="Tỷ lệ cặp đơn–mã tích lũy", title="Số lượng mua — UCI UK")
fig.tight_layout()
plt.show()

**Nhận xét:** Chỉ 28,24% cặp đơn–mã có số lượng 1; phân vị 90% của Quantity là 20.

**Đánh giá:** UCI đo Quantity trực tiếp, Olist chỉ cho đại diện số lượng qua số dòng sản phẩm. Nguồn UCI có khách sỉ nên chưa dùng nguyên các tỷ lệ này cho bán lẻ của repo.

**Đánh giá giữa nguồn:** Olist có 96,67% đơn có dòng hàng chỉ chứa một product; UCI có giỏ trung vị 15 mã và Quantity trung vị 3. Nguồn có bán sỉ và grain khác nhau giải thích giới hạn so sánh; không lấy trung bình hai nguồn hoặc loại đuôi lớn để ép cùng kết quả.

**Quy luật R02:** tách kích thước giỏ với số lượng mỗi mã; giá trị đơn hình thành từ tổng giá × số lượng rồi áp dụng giảm giá/phí theo nghiệp vụ. `basket_distribution` là tham chiếu giỏ product nhỏ; `uci_basket_distribution` và `uci_quantity_distribution` giữ vai trò đối chiếu theo invoice/mã hàng.

**Ứng dụng:** `order_items` gắn với `product_variants`, tổng tiền trong `orders`. Chưa chốt trọng số giỏ/Quantity bán lẻ, giá VND hoặc cách ánh xạ product sang variant; không nạp nguyên phân phối UCI vào repo.

## 3. Nhu cầu — sản phẩm bán chạy và nhóm hàng

**Câu hỏi:** chọn sản phẩm đều nhau có phù hợp không; chọn nhóm hàng trước có làm trùng trọng số nhu cầu không?

**Căn cứ:** [Metric 2025, S07](https://metric.vn/insights/cong-bo-bao-cao-thuong-mai-dien-tu-viet-nam-2025/) ghi nhận doanh số tập trung theo shop. Đơn vị này khác SKU, nên dùng H&M để trực tiếp đo tập trung theo sản phẩm; chưa có bằng chứng VN tương đương.

**Thực nghiệm:** dùng dòng mua H&M để tính đường tích lũy, trọng số sản phẩm và trọng số có điều kiện theo nhóm.

### 3.1. Thực nghiệm H&M — mức tập trung và trọng số sản phẩm

Đếm dòng mua theo sản phẩm trong 2019; đối chiếu đường tích lũy với chọn sản phẩm đều nhau. Mẫu chỉ gồm sản phẩm đã có dòng mua.

In [ ]:
product_counts = hm_2019.group_by("article_id").len(name="purchases").collect(
    engine="streaming",
).sort("purchases", descending=True).with_row_index("rank", offset=1)

product_counts = product_counts.with_columns(
    products_pct=100.0 * pl.col("rank") / product_counts.height,
    purchases_pct=100.0 * pl.col("purchases").cum_sum() / pl.col("purchases").sum(),
)

product_counts.filter(
    pl.col("rank").is_in([ceil(product_counts.height * share) for share in (0.01, 0.05, 0.1, 0.2)]),
).select("rank", "products_pct", "purchases_pct").with_columns(
    pl.col("products_pct", "purchases_pct").round(2),
)

**Nhận xét:** Top 1% sản phẩm chiếm 18,47% dòng mua, top 5% chiếm 46,41%; nhu cầu tập trung mạnh trước cả ngưỡng top 10%.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(product_counts["products_pct"], product_counts["purchases_pct"], label="H&M 2019")
ax.plot([0, 100], [0, 100], "--", color="gray", label="Phân bố đều")
ax.set(xlabel="% sản phẩm, xếp từ bán nhiều đến ít", ylabel="% dòng mua tích lũy")
ax.legend()
fig.tight_layout()
plt.show()

**Nhận xét:** Đường tích lũy nằm cao hơn đường phân bố đều; chọn sản phẩm đồng đều không tái hiện được mức tập trung quan sát.

**Trọng số sản phẩm:** Xác suất chọn mỗi sản phẩm bằng số dòng mua của sản phẩm chia tổng dòng mua. Bảng dưới tóm tắt theo 10 nhóm thứ hạng; trọng số từng sản phẩm vẫn giữ trong `product_weights`.

In [ ]:
product_weights = product_counts.with_columns(
    probability=pl.col("purchases") / pl.col("purchases").sum(),
    rank_decile=1 + (pl.col("rank") - 1) * 10 // product_counts.height,
)

popularity_distribution = product_weights.group_by("rank_decile").agg(
    products=pl.len(),
    purchases=pl.col("purchases").sum(),
    share_pct=100.0 * pl.col("probability").sum(),
).sort("rank_decile")

popularity_distribution.with_columns(pl.col("share_pct").round(2))

**Nhận xét:** Nhóm 10% đầu chiếm 64,04% dòng mua, trong khi nhóm 10% cuối chỉ chiếm khoảng 0,06%.

**Quy luật đề xuất:** Lấy dòng mua làm trọng số sản phẩm; dùng thứ hạng tương đối khi ánh xạ sang catalog giả lập khác quy mô.

### 3.2. Thực nghiệm H&M — nhu cầu theo nhóm hàng

Tỷ trọng trên toàn bộ dòng mua năm 2019; biểu đồ hiển thị tám nhóm lớn nhất. Phân biệt tỷ trọng nhu cầu với số sản phẩm được tạo trong catalog.

In [ ]:
hm_articles = pl.scan_csv(
    data_dir / "hm" / "articles.csv", infer_schema=False,
).select("article_id", "product_group_name").collect(engine="streaming")

group_demand = product_counts.join(
    hm_articles, on="article_id", how="left", validate="m:1",
).group_by("product_group_name").agg(
    pl.col("purchases").sum(),
).sort("purchases", descending=True).with_columns(
    share_pct=100.0 * pl.col("purchases") / pl.col("purchases").sum(),
)

group_demand.select("product_group_name", "purchases", pl.col("share_pct").round(2))

**Nhận xét:** Nhóm thân trên và thân dưới chiếm tỷ trọng mua lớn; tỷ trọng này đo nhu cầu, không đo cơ cấu số sản phẩm trong catalog.

**Chọn nhóm rồi chọn sản phẩm:** Trọng số nhóm lấy từ tổng dòng mua; trọng số sản phẩm trong nhóm lấy từ dòng mua của sản phẩm chia tổng của nhóm. Tích hai xác suất bằng trọng số sản phẩm toàn cục, tránh tính trùng ảnh hưởng nhóm hàng.

In [ ]:
group_product_weights = product_weights.select("article_id", "purchases", "probability").join(
    hm_articles, on="article_id", how="left", validate="1:1",
).with_columns(
    group_probability=pl.col("purchases").sum().over("product_group_name") / pl.col("purchases").sum(),
    product_given_group_probability=pl.col("purchases") / pl.col("purchases").sum().over("product_group_name"),
)

group_product_weights.group_by("product_group_name").agg(
    products=pl.len(),
    group_probability=pl.col("group_probability").first(),
).sort("group_probability", descending=True)

**Nhận xét:** Các nhóm nhỏ vẫn có xác suất dương; chọn nhóm rồi sản phẩm trong nhóm giữ cùng trọng số với chọn sản phẩm toàn cục.

In [ ]:
top_groups = group_demand.head(8)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    x=top_groups["share_pct"].to_numpy(), y=top_groups["product_group_name"].to_list(),
    color="#4C72B0", ax=ax,
)
ax.set(xlabel="% dòng mua", ylabel="", title="Nhóm sản phẩm — H&M 2019")
fig.tight_layout()
plt.show()

**Nhận xét:** Biểu đồ cho thấy chênh lệch nhu cầu rõ giữa các nhóm hàng lớn.

**Quy luật đề xuất:** Chọn nhóm theo tỷ trọng mua rồi chọn sản phẩm có điều kiện trong nhóm; cơ cấu tạo catalog vẫn là quyết định riêng.

**Đánh giá:** H&M bác bỏ giả định nhu cầu đồng đều trong cohort này; tỷ trọng nhóm hàng là nhu cầu mua, không phải cơ cấu catalog. Kết quả không xác định xác suất cho SKU mới hoặc chưa từng bán.

**Quy luật R03:** xác suất sản phẩm = dòng mua sản phẩm / tổng dòng mua. Có thể dùng `product_weights` hoặc chọn nhóm rồi sản phẩm từ `group_product_weights`; không nhân thêm trọng số nhóm vào xác suất toàn cục.

**Ứng dụng:** chọn sản phẩm cho `order_items`, sau đó chọn variant hợp lệ. Ánh xạ thứ hạng sang catalog giả lập khác quy mô và trọng số SKU mới là giả định cần chốt, chưa phải mức tập trung của thị trường VN.

**Đã áp dụng R03:** `distributions.product_popularity` giữ số dòng mua của mười nhóm thứ hạng H&M. Generator phân bổ đều xác suất trong mỗi nhóm, tích phần giao với khoảng thứ hạng của sản phẩm giả lập, xáo thứ hạng bằng seed rồi chọn sản phẩm trước variant. Cách chuyển catalog và chọn đều variant là giả định; không khẳng định giữ nguyên mọi tỷ trọng ở mẫu đơn cuối.

## 4. Thời điểm mua — mùa vụ, Tết và chiến dịch

**Câu hỏi:** nhu cầu thay đổi theo thời gian thế nào; phần nào có thể bổ sung riêng cho Việt Nam?

**Căn cứ:** [Ipsos Tết 2026, S05](https://www.ipsos.com/en-vn/tet-2026-vietnam-festive-hearts-sharper-carts) khảo sát 500 người 18–55 tuổi tại bốn đô thị; 53% mua phần lớn đồ Tết trước lễ 1–2 tuần. [NIQ 2025, S06](https://nielseniq.com/global/en/insights/analysis/2025/winning-vietnams-tet-holiday-optimizing-fmcg-growth-for-the-next-season/) ghi nhận mùa vụ Tết trong FMCG, không đại diện mọi ngành.

**Thực nghiệm:** dùng H&M để đo cách chuẩn hóa nhu cầu/ngày và độ biến động theo tháng; không dùng đường mùa vụ này làm đường nhu cầu VN.

### 4.1. Thực nghiệm H&M — nhu cầu theo tháng

Dùng số dòng mua/ngày để loại ảnh hưởng tháng dài/ngắn. Chỉ số 1 là bình quân ngày của năm 2019; chưa diễn giải đây là mùa vụ Việt Nam.

In [ ]:
monthly_demand = hm_2019.group_by(
    pl.col("t_dat").dt.truncate("1mo").alias("month"),
).len(name="purchases").collect(engine="streaming").sort("month")

monthly_demand = monthly_demand.with_columns(
    days=pl.col("month").dt.month_end().dt.day(),
).with_columns(
    purchases_per_day=pl.col("purchases") / pl.col("days"),
).with_columns(
    demand_index=pl.col("purchases_per_day") / (pl.col("purchases").sum() / pl.col("days").sum()),
)

monthly_demand.select(
    "month", "purchases", pl.col("purchases_per_day").round(0), pl.col("demand_index").round(2),
)

**Nhận xét:** Các tháng 10–12 đều có nhu cầu/ngày dưới mức bình quân năm; chênh lệch còn tồn tại sau khi loại ảnh hưởng độ dài tháng.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(monthly_demand["month"].dt.month(), monthly_demand["demand_index"], marker="o")
ax.axhline(1, color="gray", linestyle="--")
ax.set(xlabel="Tháng", ylabel="Chỉ số dòng mua/ngày", title="Nhu cầu theo tháng — H&M 2019")
ax.set_xticks(range(1, 13))
fig.tight_layout()
plt.show()

**Nhận xét:** Sau khi chuẩn hóa số ngày, chỉ số nhu cầu đạt khoảng 1,41 ở tháng 6 và 0,80 ở tháng 12, so với bình quân năm bằng 1.

**Quy luật đề xuất:** Cho phép nhu cầu nền thay đổi theo tháng; mùa vụ H&M 2019 chưa đại diện Việt Nam hoặc tách được tác động khuyến mãi/Tết.

### 4.2. Đánh giá và quy luật R04 cho Việt Nam

Chỉ số H&M thể hiện chênh lệch theo tháng sau chuẩn hóa độ dài tháng; một năm dữ liệu chưa tách được ảnh hưởng mùa, khuyến mại hoặc xu hướng. Ipsos bổ sung cửa sổ trước Tết nhưng 53% người trả lời không tương đương 53% tổng đơn.

**Quy luật:** đề xuất lịch Tết theo từng năm và các giai đoạn trước/trong/sau lễ; tách nhu cầu nền với chiến dịch. `monthly_demand` là tham chiếu phương pháp chuẩn hóa, chưa dùng các chỉ số tháng làm trọng số VN.

**Ứng dụng:** `orders.created_at`. Hệ số tăng đơn Tết/ngày đôi, đường theo ngày và giờ mua còn thiếu bằng chứng; chưa chốt giá trị và không nhân chồng các hiệu ứng như thể đã đo độc lập.

## 5. Giao dịch và hậu mãi

**Câu hỏi:** thanh toán, giao nhận và đánh giá liên hệ thế nào; có đủ căn cứ định lượng khuyến mại/đổi trả chưa?

**Căn cứ:** [TGM thanh toán, S04](https://tgmresearch.vn/thanh-toan-thuong-mai-dien-tu-viet-nam-cod-bnpl.html) củng cố vai trò COD nhưng đo sở thích, có thông tin mẫu 1.142/1.120 chưa thống nhất. [Milieu 2025, S08](https://mili.eu/vietnams-online-shoppers-demand-speed-and-reliability-in-2025/) nêu động lực khuyến mại và vấn đề giao nhận/đổi trả, chưa công bố đầy đủ mẫu khảo sát.

**Thực nghiệm:** dùng Olist để tách tỷ trọng giá trị thanh toán, trạng thái snapshot, thời gian giao, trễ hẹn và điểm đánh giá theo trải nghiệm.

### 5.1. Thực nghiệm Olist — trạng thái và thanh toán

Trạng thái là snapshot ghi nhận. Một đơn có thể dùng nhiều phương thức; tỷ trọng thanh toán tính theo giá trị, không ánh xạ sang COD/VietQR hoặc xác suất chuyển trạng thái.

In [ ]:
status_counts = orders_2017.group_by("order_status").len(name="orders").sort(
    "orders", descending=True,
).with_columns(share_pct=100.0 * pl.col("orders") / pl.col("orders").sum())

payment_mix = payments.join(
    orders_2017.select("order_id"), on="order_id", how="inner", validate="m:1",
).group_by("payment_type").agg(
    orders=pl.col("order_id").n_unique(),
    amount=pl.col("payment_value").sum(),
).sort("amount", descending=True).with_columns(
    value_pct=100.0 * pl.col("amount") / pl.col("amount").sum(),
)

payment_mix.select(
    "payment_type", "orders", pl.col("amount", "value_pct").round(2),
)

**Nhận xét:** Thẻ tín dụng chiếm 77,76% giá trị ghi nhận; tỷ trọng này không phải tỷ lệ đơn vì một đơn có thể có nhiều phương thức.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(
    x=status_counts["share_pct"].to_numpy(), y=status_counts["order_status"].to_list(),
    color="#4C72B0", ax=axes[0],
)
sns.barplot(
    x=payment_mix["value_pct"].to_numpy(), y=payment_mix["payment_type"].to_list(),
    color="#55A868", ax=axes[1],
)
axes[0].set(title="Trạng thái đơn — 2017", xlabel="% đơn", ylabel="", xlim=(0, 105))
axes[1].set(title="Giá trị thanh toán — 2017", xlabel="% giá trị ghi nhận", ylabel="", xlim=(0, 105))
for ax in axes:
    ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
fig.tight_layout()
plt.show()

**Nhận xét:** Phần lớn đơn trong snapshot đã giao; phân phối trạng thái không cho biết xác suất chuyển trạng thái theo thời gian.

**Đánh giá và quy luật R05:** phân phối trạng thái Olist không xác định xác suất chuyển trạng thái; tỷ trọng giá trị theo phương thức không xác định xác suất phương thức của đơn. Chưa dùng hai phân phối này làm tham số generator VN.

**Ứng dụng:** `orders`/`payments` phải theo phương thức hợp lệ của từng kênh trong schema: online có COD/VietQR; POS có cash/card/VietQR. S03 hỗ trợ bối cảnh đa kênh, không xác định tỷ trọng đơn theo kênh. Các tỷ lệ lựa chọn và thất bại thanh toán còn thiếu.

### 5.2. Thực nghiệm Olist — thời gian giao và giao trễ

Chỉ đơn `delivered` có đủ ngày và thời gian giao không âm. Trễ là ngày giao sau ngày hẹn; giao trong ngày hẹn vẫn đúng hạn.

In [ ]:
delivered_orders = order_metrics.filter(
    (pl.col("order_status") == "delivered")
    & (pl.col("order_delivered_customer_date") >= pl.col("order_purchase_timestamp"))
    & pl.col("order_estimated_delivery_date").is_not_null(),
).with_columns(
    delivery_days=(
        pl.col("order_delivered_customer_date") - pl.col("order_purchase_timestamp")
    ).dt.total_seconds() / 86400,
    delay_days=(
        pl.col("order_delivered_customer_date").dt.date()
        - pl.col("order_estimated_delivery_date").dt.date()
    ).dt.total_days(),
    late=(
        pl.col("order_delivered_customer_date").dt.date()
        > pl.col("order_estimated_delivery_date").dt.date()
    ),
)

delivered_orders.select("delivery_days", "delay_days").describe()

**Nhận xét:** Thời gian giao trung bình 13,03 ngày cao hơn trung vị 11,03; delay_days âm biểu thị giao sớm hơn ngày hẹn.

In [ ]:
delivered_orders.select(
    orders=pl.len(),
    late_pct=(100.0 * pl.col("late").mean()).round(2),
)

**Nhận xét:** 5,65% trong 43.426 đơn giao hợp lệ bị trễ; mẫu số không bao gồm đơn chưa giao hoặc thiếu ngày.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.ecdfplot(x=delivered_orders["delivery_days"].to_numpy(), ax=ax)
ax.set(xlabel="Thời gian giao (ngày)", ylabel="Tỷ lệ đơn tích lũy", title="Thời gian giao — Olist 2017")
fig.tight_layout()
plt.show()

**Nhận xét:** Đường tích lũy có đuôi phải dài, cho thấy một số đơn giao chậm hơn nhiều so với phần lớn đơn.

**Quy luật đề xuất:** Sinh thời gian giao thực tế và ngày hẹn riêng để tạo đơn đúng hẹn/trễ; cần thích nghi theo nghiệp vụ giao nhận VN.

### 5.3. Thực nghiệm Olist — điểm đánh giá theo giao nhận

Chỉ đơn đủ điều kiện giao hàng và có đánh giá; điểm là trung bình trong từng đơn. Đây là liên hệ quan sát, không kết luận nhân quả. Phân phối điểm 1–5 bên dưới dùng từng phiếu đánh giá.

In [ ]:
review_comparison = delivered_orders.filter(pl.col("review_score").is_not_null()).group_by(
    "late",
).agg(
    orders=pl.len(),
    mean_score=pl.col("review_score").mean(),
    low_score_orders_pct=100.0 * (pl.col("review_score") <= 2).mean(),
).sort("late").with_columns(
    pl.col("mean_score", "low_score_orders_pct").round(2),
)

review_comparison

**Nhận xét:** Tỷ lệ điểm trung bình/đơn ≤2 tăng từ 9,16% ở nhóm đúng hẹn lên 61,49% ở nhóm trễ; hai nhóm có trải nghiệm khác rõ.

**Xác suất điểm 1–5 theo giao nhận:** Đếm từng phiếu đánh giá hợp lệ của các đơn giao đủ điều kiện; mẫu số là số phiếu trong mỗi nhóm đúng hẹn/trễ. Phép đo này khác điểm trung bình mỗi đơn ở bảng trên.

In [ ]:
rating_counts = reviews.filter(pl.col("review_score").is_in([1.0, 2.0, 3.0, 4.0, 5.0])).join(
    delivered_orders.select("order_id", "late"), on="order_id", how="inner", validate="m:1",
).group_by("late", "review_score").len(name="reviews")

rating_distribution = pl.DataFrame({"late": [False, True]}).join(
    pl.DataFrame({"review_score": [1.0, 2.0, 3.0, 4.0, 5.0]}), how="cross",
).join(
    rating_counts, on=["late", "review_score"], how="left", validate="1:1",
).with_columns(
    pl.col("reviews").fill_null(0),
).with_columns(
    probability=pl.col("reviews") / pl.col("reviews").sum().over("late"),
).sort("late", "review_score")

rating_distribution

**Nhận xét:** Ở cấp phiếu đánh giá, tỷ lệ 5 điểm là 61,53% cho nhóm đúng hẹn, so với 17,90% cho nhóm trễ.

**Quy luật đề xuất:** Chọn điểm 1–5 theo xác suất của nhóm giao nhận. Đây là liên hệ quan sát; đánh giá đơn chưa tương đương đánh giá sản phẩm.

**Đánh giá và quy luật R06:** đề xuất thời gian giao có phân phối và điểm đánh giá có điều kiện theo trải nghiệm giao nhận. `rating_distribution` mô tả phiếu đánh giá của đơn Olist, chưa xác định xác suất khách viết review hoặc điểm review của từng item.

**Ứng dụng:** `shipments` và `product_reviews`; cần ánh xạ review theo đơn sang review theo item và quyết định cách biểu diễn ngày hẹn trước khi triển khai. Trung vị 11,03 ngày và tỷ lệ trễ 5,65% chỉ thuộc Brazil; chưa chốt thời gian, tỷ lệ trễ hoặc mức ảnh hưởng tại VN.

### 5.4. Khuyến mại và đổi trả — phần chưa đủ dữ liệu (R07)

S08 gợi ý giảm giá, phí giao hàng và hàng lỗi/sai là các yếu tố đáng mô phỏng. Đây là căn cứ chọn tình huống, chưa phải tỷ lệ dùng voucher hoặc hoàn hàng trên tổng đơn. Các cohort EDA hiện chưa đo được các tỷ lệ phù hợp cho repo; không suy từ invoice hủy UCI thành tỷ lệ hoàn hàng.

**Ứng dụng:** `coupons`, `coupon_redemptions`, `return_requests`, `return_items`, `refunds`. Giữ các quy trình nghiệp vụ; tỷ lệ kích hoạt, giá trị giảm và lý do hiện giữ theo kịch bản cũ dưới nhãn giả định, chưa được xem là ước lượng từ nghiên cứu VN. Không thêm mô hình dự đoán để lấp phần thiếu này.

## 6. Kết quả nghiên cứu và quy luật đề xuất cho generator

**Kết quả đo** thuộc cohort nguồn; **đề xuất** là lựa chọn thiết kế; **chưa đủ căn cứ** nghĩa là chưa có ước lượng phù hợp cho shop mô phỏng. R03 đã áp dụng trọng số theo nhóm thứ hạng; các giá trị khác giữ từ cấu hình cũ với nhãn giả định, chưa thay bằng các phân phối nước ngoài hoặc tỷ lệ khảo sát VN.

| Hành vi | Kết quả thực nghiệm | Căn cứ Việt Nam | Quy luật/cách lấy mẫu đề xuất | Giới hạn, tham số chưa chốt | Dữ liệu ứng dụng |
|---|---|---|---|---|---|
| R01. Mua lại | UCI: 63,99% khách có ≥2 đơn; H&M: 66,20% có ≥2 ngày mua | S02: tần suất online nhiều nơi bán | Giữ riêng phân phối số đơn UCI và ngày mua H&M; khoảng mua có điều kiện theo nhóm | Tham chiếu; chưa chốt tỷ trọng nhóm/số đơn tại shop VN | `customers`, `orders.created_at` |
| R02. Giỏ hàng | Olist: 96,67% đơn có dòng hàng chỉ chứa một product; UCI: trung vị 15 mã/đơn | S01: khác biệt chi tiêu; S07: giá theo ngành | Chọn kích thước giỏ theo phân phối nguồn phù hợp; số lượng lấy riêng; tổng tiền tính từ dòng hàng | Chưa chốt nguồn/trọng số áp dụng; product/StockCode chưa phải variant; giá VND và Quantity bán lẻ còn thiếu | `order_items`, `orders` |
| R03. Sản phẩm | H&M: top 10% sản phẩm có bán chiếm 64,04% dòng mua | S07 chỉ đo tập trung theo shop | Đã áp dụng trọng số mười nhóm thứ hạng H&M; chọn sản phẩm rồi variant | Phân bổ đều trong nhóm thứ hạng, ánh xạ catalog và chọn variant là giả định; chưa phải phân phối SKU Việt Nam | `products`, `product_variants`, `order_items` |
| R04. Thời điểm | H&M: chỉ số tháng 6 ≈1,41; tháng 12 ≈0,80 | S05: 53% mua chính trước Tết 1–2 tuần; S06 giới hạn FMCG | Đề xuất lịch Tết theo năm; phân biệt nhu cầu nền và chiến dịch | Chưa chốt đường mùa vụ VN, hệ số ngày đôi/Tết và phân phối giờ | `orders.created_at` |
| R05. Thanh toán/kênh | Olist: thẻ tín dụng chiếm 77,76% giá trị, không phải số đơn | S03: người mua đa kênh; S04: ưu tiên COD | Giữ các kênh/phương thức hợp lệ theo schema; chưa lấy tỷ lệ khảo sát làm tỷ lệ đơn | Tỷ trọng online/POS, COD/VietQR và thất bại thanh toán còn thiếu | `orders.channel`, `orders.payment_method`, `payments` |
| R06. Giao nhận/review | Olist: trung vị 11,03 ngày, trễ 5,65%; điểm 5 là 61,53%/17,90% theo đúng hẹn/trễ | S08: vấn đề giao nhận được người mua quan tâm | Đề xuất sinh thời gian giao, xác định trải nghiệm rồi chọn điểm có điều kiện | Tham chiếu Brazil; thiếu thời gian VN, ngày hẹn và xác suất viết review; phiếu theo đơn khác review theo item | `shipments`, `product_reviews` |
| R07. Khuyến mại/đổi trả | Chưa đo tỷ lệ dùng voucher hoặc hoàn hàng phù hợp từ các cohort đã phân tích | S08 định hướng động lực và lý do | Giữ các giá trị kịch bản cũ dưới nhãn giả định | Tỷ lệ toàn đơn và phân phối lý do đầy đủ còn thiếu | `coupons`, `coupon_redemptions`, `return_requests`, `return_items`, `refunds` |

S01–S11 là mã nguồn trong [báo cáo nghiên cứu Việt Nam](../../docs/project/GENERATOR_VIETNAM_RESEARCH.md); căn cứ được dẫn trực tiếp trong từng nhóm phân tích.

**Áp dụng vào dự án:** dùng bảng trên làm đầu vào thiết kế generator. Quy luật có trọng số phải chỉ rõ biến phân phối đã đo, đơn vị và cách ánh xạ; tham số chưa có ước lượng phù hợp giữ theo kịch bản cũ và được ghi rõ là giả định. Danh mục giá trị và ý nghĩa nằm trong [GENERATOR_METRICS](../../docs/project/GENERATOR_METRICS.md).

**Giới hạn:** không trộn các nguồn thành một phân phối Việt Nam, không sao chép mức bán sỉ sang bán lẻ và không biến tương quan thành nhân quả. Dữ liệu sinh theo quy luật là đầu ra của phương pháp; chưa bổ sung benchmark generator trước/sau.

## Phụ lục. Hành trình mua — tham khảo ngoài phạm vi sinh SQL

**Câu hỏi:** phiên có sự kiện mua khác thế nào với chuỗi xem → thêm giỏ → mua đúng thứ tự?

Giữ phân tích eCommerce Behavior để tham khảo hành trình và khoảng chờ. Không dùng purchase event làm số đơn, thời gian giữa event làm HTTP latency hoặc tỷ lệ funnel làm tỷ lệ hủy đơn. Phần này chưa chuyển thành quy luật sinh SQL và không khôi phục chức năng sinh log.

In [ ]:
behavior_logs = pl.scan_csv(
    [
        data_dir / "ecommerce_behavior" / "2019-Oct.csv",
        data_dir / "ecommerce_behavior" / "2019-Nov.csv",
    ],
    infer_schema=False,
).select(
    "event_time", "event_type", "user_id", "user_session", "product_id",
    "price", "brand", "category_code",
).with_columns(
    pl.col("event_time").str.strip_suffix(" UTC").str.to_datetime(
        "%Y-%m-%d %H:%M:%S", time_zone="UTC", strict=False,
    ),
    pl.col("price").cast(pl.Float64, strict=False),
)

In [ ]:
behavior_logs.select("price").collect(engine="streaming").describe()

**Nhận xét:** Giá sự kiện có giá trị 0 và lệch phải; giá gắn với một lượt tương tác chưa phải doanh thu đơn hàng.

In [ ]:
behavior_logs.null_count().collect(engine="streaming")

**Nhận xét:** Brand và category_code thiếu nhiều, trong khi session thiếu 12 giá trị; không cần loại dòng chỉ vì thiếu metadata sản phẩm.

### A.1. Phiên truy cập

Khóa phiên là `user_id` + `user_session`. Quan sát 1/10–1/11/2019 UTC; bỏ phiên chạm ngày đầu/cuối cửa sổ để hạn chế chuỗi bị cắt ở biên.

`duration_minutes` là khoảng cách giữa sự kiện đầu/cuối quan sát, không phải thời gian thực ở lại trang.

In [ ]:
session_key = ["user_id", "user_session"]
path_key = session_key + ["product_id"]

observed_events = behavior_logs.select(
    *path_key, "event_time", "event_type",
).filter(
    pl.col("user_id").is_not_null()
    & pl.col("user_session").is_not_null()
    & pl.col("product_id").is_not_null()
    & pl.col("event_time").is_between(
        pl.datetime(2019, 10, 1, time_zone="UTC"),
        pl.datetime(2019, 11, 2, time_zone="UTC"), closed="left",
    ),
)

session_stats = observed_events.group_by(session_key).agg(
    start=pl.col("event_time").min(),
    end=pl.col("event_time").max(),
    events=pl.len(),
    viewed_products=pl.col("product_id").filter(pl.col("event_type") == "view").n_unique(),
    has_view=(pl.col("event_type") == "view").any(),
    has_cart=(pl.col("event_type") == "cart").any(),
    has_purchase=(pl.col("event_type") == "purchase").any(),
).filter(
    (pl.col("start") >= pl.datetime(2019, 10, 2, time_zone="UTC"))
    & (pl.col("end") < pl.datetime(2019, 11, 1, time_zone="UTC")),
).with_columns(
    duration_minutes=(pl.col("end") - pl.col("start")).dt.total_seconds() / 60,
).collect(engine="streaming")

session_events = observed_events.join(
    session_stats.lazy().select(session_key), on=session_key, how="semi",
)

session_stats.select("events", "viewed_products", "duration_minutes").describe()

**Nhận xét:** Phiên điển hình có 2 sự kiện và xem 1 sản phẩm; khoảng quan sát trung bình cao hơn nhiều trung vị do một số session kéo dài.

In [ ]:
session_stats.select(
    sessions=pl.len(),
    with_cart_pct=(100.0 * pl.col("has_cart").mean()).round(2),
    with_purchase_pct=(100.0 * pl.col("has_purchase").mean()).round(2),
)

**Nhận xét:** Tỷ lệ phiên có purchase 6,83% cao hơn có cart 6,27%; sự hiện diện đơn lẻ không thể thay cho funnel có thứ tự.

In [ ]:
event_mix = session_events.group_by("event_type").len(name="events").collect(
    engine="streaming",
).sort("events", descending=True)

event_mix

**Nhận xét:** View chiếm phần lớn sự kiện; thêm giỏ và mua xuất hiện ít hơn đáng kể.

**Quy luật đề xuất:** Nếu mô phỏng hành vi, giữ người dùng/phiên/sản phẩm xuyên hành trình thay vì chọn từng request độc lập.

### A.2. Funnel xem → thêm giỏ → mua

Cùng sản phẩm trong cùng phiên, timestamp tăng nghiêm ngặt. Mẫu số là phiên có `view`; khác với tỷ lệ phiên chỉ cần có cart/purchase.

Mỗi đường đi lấy view đầu tiên, cart đầu tiên sau view và purchase đầu tiên sau cart.

In [ ]:
cart_events = session_events.filter(pl.col("event_type") == "cart").select(
    *path_key, "event_time",
).collect(engine="streaming")
cart_products = cart_events.select(path_key).unique()

view_times = session_events.filter(pl.col("event_type") == "view").join(
    cart_products.lazy(), on=path_key, how="semi",
).group_by(path_key).agg(
    view_at=pl.col("event_time").min(),
).collect(engine="streaming")

cart_candidates = cart_events.join(view_times, on=path_key, how="inner")
cart_paths = cart_candidates.filter(pl.col("event_time") > pl.col("view_at")).group_by(
    path_key,
).agg(
    view_at=pl.col("view_at").first(),
    cart_at=pl.col("event_time").min(),
)

In [ ]:
purchase_events = session_events.filter(pl.col("event_type") == "purchase").select(
    *path_key, "event_time",
).collect(engine="streaming")

purchase_candidates = purchase_events.join(cart_paths, on=path_key, how="inner")
purchase_paths = purchase_candidates.filter(
    pl.col("event_time") > pl.col("cart_at"),
).group_by(path_key).agg(
    view_at=pl.col("view_at").first(),
    cart_at=pl.col("cart_at").first(),
    purchase_at=pl.col("event_time").min(),
)

funnel = pl.DataFrame({
    "stage": ["view", "view → cart", "view → cart → purchase"],
    "sessions": [
        session_stats["has_view"].sum(),
        cart_paths.select(session_key).unique().height,
        purchase_paths.select(session_key).unique().height,
    ],
}).with_columns(
    conversion_pct=100.0 * pl.col("sessions") / pl.col("sessions").first(),
)

funnel.with_columns(pl.col("conversion_pct").round(2))

**Nhận xét:** Trong phiên có view, 6,23% đạt thêm giỏ và 3,10% đạt đủ ba bước trên cùng sản phẩm; mỗi tầng đếm phiên distinct.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
sns.barplot(
    x=funnel["conversion_pct"].to_numpy(), y=funnel["stage"].to_list(),
    color="#4C72B0", ax=ax,
)
ax.bar_label(ax.containers[0], fmt="%.2f%%", padding=3)
ax.set(xlabel="% phiên có view", ylabel="", title="Funnel có thứ tự — 10/2019", xlim=(0, 110))
fig.tight_layout()
plt.show()

**Nhận xét:** Chỉ một phần nhỏ phiên có view đi qua đủ chuỗi được ghi nhận; không nên gán mọi phiên thiếu bước thành bỏ giỏ.

Các đường đi có cart trùng giờ view đầu tiên hoặc purchase trùng giờ cart đầu tiên được đếm riêng dưới đây; timestamp đó không chứng minh được thứ tự.

In [ ]:
pl.DataFrame({
    "same_timestamp": ["view/cart", "cart/purchase"],
    "session_product_paths": [
        cart_candidates.filter(pl.col("event_time") == pl.col("view_at")).select(path_key).unique().height,
        purchase_candidates.filter(pl.col("event_time") == pl.col("cart_at")).select(path_key).unique().height,
    ],
})

**Nhận xét:** Có 2.623 đường đi trùng timestamp view/cart; các timestamp bằng nhau không đủ chứng minh thứ tự hành động.

**Quy luật đề xuất:** Chỉ nhận chuỗi có timestamp tăng nghiêm ngặt; không dùng tỷ lệ phiên làm trọng số request hoặc KPI mart hiện tại.

### A.3. Thời gian giữa các bước

Tính trên đường đi `(khách, phiên, sản phẩm)` hoàn thành đủ ba bước; không đại diện nhóm bỏ giỏ.

In [ ]:
conversion_times = purchase_paths.select(
    view_to_cart_minutes=(pl.col("cart_at") - pl.col("view_at")).dt.total_seconds() / 60,
    cart_to_purchase_minutes=(pl.col("purchase_at") - pl.col("cart_at")).dt.total_seconds() / 60,
    view_to_purchase_minutes=(pl.col("purchase_at") - pl.col("view_at")).dt.total_seconds() / 60,
)

conversion_times.describe()

**Nhận xét:** Trung vị xem–giỏ là 0,45 phút, giỏ–mua là 1,35 phút; thời gian chờ khác nhau giữa các bước.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.ecdfplot(x=conversion_times["view_to_purchase_minutes"].to_numpy(), ax=ax)
ax.set_xscale("log")
ax.set(xlabel="Phút từ xem đến mua (thang log)", ylabel="Tỷ lệ đường đi tích lũy", title="Thời gian chuyển đổi — 10/2019")
fig.tight_layout()
plt.show()

**Nhận xét:** Xem–mua có trung vị 2,18 phút nhưng trung bình 4,67 phút; một số đường đi kéo dài tạo đuôi phải.

**Quy luật đề xuất:** Sinh khoảng chờ có phân phối cho nhóm hoàn thành; không dùng thời gian hành động làm HTTP latency hoặc đại diện nhóm bỏ giỏ.